#  TypedDict的使用

In [1]:
from itertools import tee
from tempfile import tempdir
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
)

In [6]:
from typing_extensions import Annotated,TypedDict


class MovieTypedDict(TypedDict):
    """电影的信息"""
    title : Annotated[str,"电影的名称"]
    year : Annotated[int,"电影的上映时间，四位数"]
    director : Annotated[str,"电影的导演"]
    rating : Annotated[float,"电影的评分，满分是10分，可以包含一位小数"]



structured_model = model.with_structured_output(MovieTypedDict,method="function_calling")
response = structured_model.invoke("给我介绍一下电影《星际穿越》")
print(response)
print(type(response))

{'title': '星际穿越', 'director': '克里斯托弗·诺兰', 'year': 2014, 'rating': 9.4}
<class 'dict'>


# 嵌套结构的使用

In [8]:
from typing import List

class Actor(TypedDict):
    """演员的信息"""
    name : Annotated[str,"演员的名字"]
    role : Annotated[str,"扮演的角色"]


class MovieTypedDict(TypedDict):
    """电影的信息"""
    title : Annotated[str,"电影的名称"]
    year : Annotated[int,"电影的上映时间，四位数"]
    director : Annotated[str,"电影的导演"]
    rating : Annotated[float,"电影的评分，满分是10分，可以包含一位小数"]
    cast : Annotated[List[Actor],"演员的列表"]  # 嵌套列表的结构


structured_model = model.with_structured_output(MovieTypedDict,method="function_calling")
response = structured_model.invoke("给我介绍一下电影《星际穿越》")
print(response)
print(type(response))

{'title': '星际穿越', 'director': '克里斯托弗·诺兰', 'year': 2014, 'rating': 9.4, 'cast': [{'name': '马修·麦康纳', 'role': '库珀'}, {'name': '安妮·海瑟薇', 'role': '布兰德博士'}, {'name': '杰西卡·查斯坦', 'role': '成年墨菲'}, {'name': '迈克尔·凯恩', 'role': '布兰德教授'}, {'name': '麦肯吉·弗依', 'role': '少年墨菲'}, {'name': '马特·达蒙', 'role': '曼恩博士'}]}
<class 'dict'>


In [9]:
class MovieDict(TypedDict):
    """电影的信息"""
    title : Annotated[str,...,"电影的名称"]
    year : Annotated[int,...,"电影的上映时间，四位数"]
    director : Annotated[str,...,"电影的导演"]
    rating : Annotated[float,...,"电影的评分，满分是10分，可以包含一位小数"]


structured_model = model.with_structured_output(MovieDict,method="function_calling")
response = structured_model.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")
print(response)

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 0}
